In [0]:
df = spark.sql("FROM supply_chain_live.bronze.raw_supply_chain")
df.display()

In [0]:
metadata = spark.sql("FROM supply_chain_live.bronze.metadata")
metadata.display()

In [0]:
df.select(
    "Customer Email",
    "Customer Country",
    "Benefit per order",
    "Shipping date (DateOrders)",
).limit(10).display()

In [0]:
df.select("Product Description").distinct().display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum


# Count the number of null values in each column
null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)
# Only keep null values in a dictionary
null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
import re

# Customer Email -> customer_email
def to_snake_case(name):
    return re.sub(r"[\s]+", "_", name.strip().casefold())

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

to_snake_case("Customer          Email   AcCount")

In [0]:
df_cleaned_columns = rename_columns_to_snake_case(df)
df_cleaned_columns.display()

In [0]:
df_cleaned_columns.select("shipping_date_(dateorders)").limit(1).display()

In [0]:
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when

# 6/19/2017 4:41
df_ cleaned = df_cleaned_columns.withColumn(
    "shipping_date", to_timestamp("shipping_date_(dateorders)", "M/d/yyyy H:m")
).withColumn(
    "order_zipcode", coalesce(col("order_zipcode").cast("string"), lit("unknown"))
).withColumn(
    "customer_zipcode",
    coalesce(col("customer_zipcode").cast("string"), lit("unknown")),
).withColumn(
    "customer_country",
    when(col("customer_country") == "EE. UU.", "United States").otherwise(
        col("customer_country")
    ),
).withColumn(
    "order_date", to_timestamp("order_date_(dateorders)", "M/d/yyyy H:m")
).drop("order_date_(dateorders)", "shipping_date_(dateorders)", "product_description", "customer_email", "customer_password"

df_cleaned.select("customer_country", "order_date", "shipping_date").display()